# Notebook 1 — Cryogenic DR Thermal Model

This notebook demonstrates the lumped-RC ODE model of a dilution refrigerator (DR)
cool-down, calibrated to the **Oxford Instruments Triton 400** (~400 µW at 15 mK).

## Six-stage model
| # | Stage | Nominal T |
|---|---|---|
| 1 | 300 K Plate (fixed boundary) | 300 K |
| 2 | 50 K Pulse-tube stage | ~44 K |
| 3 | 4 K Pulse-tube stage | ~3.6 K |
| 4 | Still | ~480 mK |
| 5 | Cold Plate | ~33 mK |
| 6 | Mixing Chamber (MXC) | ~13 mK |

## ODE
$$C_i \frac{dT_i}{dt} = G_i (T_{i-1} - T_i) - G_{i+1}(T_i - T_{i+1}) - P_{\text{cool},i}(T_i) + P_{\text{qubit},i}$$

Cooling power fits (Oxford Triton 400 specs + Pobell *Matter and Methods at Low Temperatures*):
- **PT stage 1 (50 K)**: $P = 40\left(1 - 30/T\right)$ W (heat-pump law)
- **PT stage 2 (4 K)**: $P = 1.5\left(1 - 2.5/T\right)$ W
- **Still**: $P = 40.8 \times 10^{-3}\, T^2$ W
- **Cold Plate**: $P = 0.2\, T^2$ W
- **MXC**: $P = 0.04\, T^2$ W (³He dilution, $\dot{n}_3 = 300\,\mu\text{mol/s}$)

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt
from cryo_thermal import simulate_cooldown, STAGE_NAMES, _G0, _C

print('Stage names:', STAGE_NAMES)
print('Conductances G [W/K]:', _G0)
print('Capacities  C [J/K]:', _C)
print('Time constants τ = C/G [h]:', (_C[1:] / _G0[1:]) / 3600)

In [ ]:
# Run the 10-hour cool-down simulation
t_h, T = simulate_cooldown(t_hours=10.0, n_eval=3000)

print(f'Simulation: {len(t_h)} time points, t = 0 → {t_h[-1]:.2f} h')
print()
print('Steady-state temperatures (last point):')
for i, (name, temp) in enumerate(zip(STAGE_NAMES, T[-1])):
    if temp > 1:
        print(f'  {name:20s}: {temp:.2f} K')
    elif temp > 0.001:
        print(f'  {name:20s}: {temp*1e3:.1f} mK')
    else:
        print(f'  {name:20s}: {temp*1e6:.1f} µK')

In [ ]:
# ── Full cool-down curves (linear scale) ─────────────────────────────────────
colors = ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3','#a65628']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: linear scale (first 5 stages + 300K)
ax = axes[0]
for i in range(6):
    ax.plot(t_h, T[:, i], color=colors[i], lw=2, label=STAGE_NAMES[i])
ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('Temperature (K)', fontsize=12)
ax.set_title('DR Cool-down — All Stages (Linear)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: log scale to see sub-K stages
ax = axes[1]
for i in range(6):
    ax.semilogy(t_h, T[:, i], color=colors[i], lw=2, label=STAGE_NAMES[i])
ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('Temperature (K)', fontsize=12)
ax.set_title('DR Cool-down — All Stages (Log Scale)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/cooldown_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/cooldown_full.png')

In [ ]:
# ── Sub-K stages zoom ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for i in [3, 4, 5]:
    ax.semilogy(t_h, T[:, i] * 1e3, color=colors[i], lw=2.5, label=STAGE_NAMES[i])

ax.axhline(15, color='black', ls='--', lw=1.5, alpha=0.6, label='MXC target 15 mK')
ax.axhline(30, color='gray',  ls=':',  lw=1.5, alpha=0.6, label='CP target 30 mK')

ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('Temperature (mK)', fontsize=12)
ax.set_title('Sub-Kelvin Stage Cool-down', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/subkelvin_cooldown.png', dpi=150, bbox_inches='tight')
plt.show()

T_ss = T[-1]
print(f'\nFinal temperatures:')
print(f'  Still      : {T_ss[3]*1e3:.1f} mK')
print(f'  Cold Plate : {T_ss[4]*1e3:.1f} mK')
print(f'  MXC        : {T_ss[5]*1e3:.2f} mK  (target: ~15 mK)')

In [ ]:
# ── Cooling power curves vs temperature ──────────────────────────────────────
from cryo_thermal import cooling_power

T_50K_range = np.linspace(30.1, 300, 500)
T_4K_range  = np.linspace(2.51, 20, 500)
T_mK_range  = np.linspace(0.001, 1.0, 500)

P_50K = np.array([cooling_power(1, T) for T in T_50K_range])
P_4K  = np.array([cooling_power(2, T) for T in T_4K_range])
P_still = np.array([cooling_power(3, T) for T in T_mK_range])
P_cp    = np.array([cooling_power(4, T) for T in T_mK_range])
P_mxc   = np.array([cooling_power(5, T) for T in T_mK_range])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(T_50K_range, P_50K, 'r-', lw=2, label='50 K PT stage')
axes[0].plot(T_4K_range, P_4K, 'b-', lw=2, label='4 K PT stage')
axes[0].set_xlabel('Temperature (K)'); axes[0].set_ylabel('P_cool (W)')
axes[0].set_title('Pulse Tube Cooling Power'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(T_mK_range * 1e3, P_still * 1e3, 'g-', lw=2, label='Still')
axes[1].set_xlabel('Temperature (mK)'); axes[1].set_ylabel('P_cool (mW)')
axes[1].set_title('Still Cooling Power (T² law)'); axes[1].legend(); axes[1].grid(alpha=0.3, which='both')

axes[2].semilogy(T_mK_range * 1e3, P_cp * 1e6, 'm-', lw=2, label='Cold Plate')
axes[2].semilogy(T_mK_range * 1e3, P_mxc * 1e6, 'k-', lw=2, label='MXC')
axes[2].set_xlabel('Temperature (mK)'); axes[2].set_ylabel('P_cool (µW)')
axes[2].set_title('CP and MXC Cooling Power (T² law)'); axes[2].legend(); axes[2].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../outputs/cooling_power_curves.png', dpi=150, bbox_inches='tight')
plt.show()